# test-writer: замер эффекта навыка

Один и тот же агент пишет тесты к 10 модулям в двух условиях:
- **baseline** — без навыка (может писать и запускать тесты);
- **skill** — с навыком `test-writer` (инструкция, справочники и скрипт mutation testing).

Качество тестов измеряется mutation score: сколько внесённых в код багов тесты ловят.

Два варианта запуска:
- **A. Groq API** — бесплатно, большая модель `gpt-oss-120b`. Видеокарта не нужна.
- **B. Локальная модель** на GPU Colab — без ключа, но медленнее.

## 1. Код и зависимости
Впиши адрес своего репозитория.

In [ ]:
REPO = "https://github.com/sofetizm/test-writer.git"
%cd /content
!git clone -q $REPO test-writer || (cd test-writer && git pull -q)
%cd /content/test-writer
!pip install -q -r requirements.txt && pip install -q -e .

## 2. Google Drive (рекомендуется)
Результаты пишутся на Диск, поэтому если Colab отключится, просто запусти ячейку с прогоном
ещё раз: готовые задачи пропустятся, и прогон продолжится с того же места.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/test-writer-results'

## 3A. Прогон через Groq (бесплатно)
Ключ — в секретах Colab под именем `GROQ_API_KEY` (значок 🔑 слева).

Бесплатный тариф ограничивает число токенов в день, а полный прогон (10 задач × 2 условия)
занимает больше дневного лимита. Если появятся ошибки про лимит, остановись и запусти
ячейку завтра: прогон продолжится с того места, где остановился.

In [ ]:
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('GROQ_API_KEY')
os.environ['OPENAI_BASE_URL'] = 'https://api.groq.com/openai/v1'

!python -m twbench.run --backend openai --model openai/gpt-oss-120b --name gpt-oss-120b-v0.2 \
    --conditions baseline,skill --reasoning-effort low --max-tokens 3000 --rpm 4 \
    --price-in 0.15 --price-out 0.60 --out "$OUT"

### (Опционально) Абляция: только текст навыка, без скрипта
Показывает, какая часть эффекта даёт инструкция, а какая — обратная связь от mutation testing.

In [ ]:
!python -m twbench.run --backend openai --model openai/gpt-oss-120b --name gpt-oss-120b-v0.2 \
    --conditions skill-text --reasoning-effort low --max-tokens 3000 --rpm 4 \
    --price-in 0.15 --price-out 0.60 --out "$OUT"

## 3B. Локальная модель (без ключа)
Нужна видеокарта: Среда выполнения → Сменить среду выполнения → T4 GPU.
Медленнее, чем API: примерно 2–4 часа на все задачи. Можно взять часть задач через `--tasks`.

In [ ]:
!pip install -q -e ".[local]"
!python -m twbench.run --backend hf --model Qwen/Qwen2.5-Coder-7B-Instruct --name qwen2.5-coder-7b \
    --conditions baseline,skill --max-tokens 2500 --out "$OUT" \
    --tasks pagination,money,intervals,retry_backoff,durations

## 4. Отчёт
Сначала пересчёт оценок по сохранённым тестам: модель не вызывается, лимит не тратится.
Нужен, если код оценки обновился после прогона.

In [ ]:
!python -m twbench.rescore --results "$OUT"

In [ ]:
!python -m twbench.report --results "$OUT"
from IPython.display import Image, Markdown, display
display(Image(OUT + '/scores.png'))

## 5. Скачать результаты
Папку `results` из архива положи в репозиторий.

In [ ]:
!cd "$OUT" && zip -qr /content/results.zip . -x "*/ws/*"
from google.colab import files
files.download('/content/results.zip')